# Face Anonymization from 5 Given Landmarks (cedalion-only)

Pure-geometric face-removal pipeline for Einstar scans. Takes the 5 standard
anatomical landmarks (Nz, Iz, Cz, Lpa, Rpa) as input and deletes the facial
region without any ML / face-detection dependency.

**Pipeline:**
1. Load scan
2. Pick 5 landmarks interactively via `cedalion.plots.plot_surface(pick_landmarks=True)`
3. Wrap into a `LabeledPointCloud`
4. Normalize axes (X=up, Y=anterior, Z=left) and isolate head
5. Build face-region mask purely from the 5 landmarks
6. Visualize the mask
7. Delete masked vertices (no smoothing)
8. Validate result

This is the cedalion-only alternative to notebook 47's MediaPipe pipeline and
matches the exposé contract: NAS/LPA/RPA assumed given, no DL face detection.

In [ ]:
import logging

import numpy as np
import pyvista as pv
import matplotlib.pyplot as plt
import xarray as xr

import cedalion
import cedalion.dataclasses as cdc
import cedalion.io
import cedalion.plots
from cedalion.geometry.photogrammetry.anonymization import (
    normalize_axes,
    isolate_head,
    align_axes_from_landmarks,
    detect_cap_boundary,
    face_mask_from_landmarks,
    delete_masked_vertices,
    validate_anonymization,
)
from cedalion.vtktutils import trimesh_to_vtk_polydata
from cedalion.dataclasses import VTKSurface

logging.getLogger('cedalion').setLevel(logging.WARNING)
pv.set_jupyter_backend('server')

# === CONFIG ===
SUBJECT_NUMBER = 17
SCANS_FOLDER = '/home/ma7/BA/PG_Subjects'

# True: run the picker. False: use cached landmarks (fill after first pick).
INTERACTIVE = True

# Mask tunables (mm)
LANDMARK_KEEP_RADIUS = 8.0      # preserve a small sphere around each landmark
EAR_DELETE_RADIUS_MM = 40.0     # radius of the delete sphere around Lpa / Rpa

# Cap boundary detection tunables
CAP_BAND_WIDTH_MM = 15.0
CAP_BIN_SIZE_MM = 2.0
CAP_FOOT_GRAD_THRESHOLD = 0.2

## 1. Load the Einstar scan

In [16]:
path = f'{SCANS_FOLDER}/Subject{SUBJECT_NUMBER}/Subject{SUBJECT_NUMBER}.obj'
surface = cedalion.io.read_einstar_obj(path)
print(f'Loaded: {surface.nvertices:,} vertices, {surface.nfaces:,} faces')

Loaded: 1,284,667 vertices, 2,223,716 faces


## 2. Pick the 5 landmarks interactively

Uses cedalion's built-in picker (`cedalion.plots.plot_surface(..., pick_landmarks=True)`,
authored by Mariia Iudina, upstream cedalion). Right-click on the mesh to place a
sphere; click again on the sphere to cycle its label through
`Nz -> Iz -> Cz -> Lpa -> Rpa`, or cycle past `Rpa` back to `Nz` to remove it.

Close the window when all 5 are placed with correct labels.

In [17]:
if INTERACTIVE:
    pvplt = pv.Plotter()
    get_landmarks = cedalion.plots.plot_surface(
        pvplt, surface, opacity=1.0, pick_landmarks=True
    )
    pvplt.show()

Widget(value='<iframe src="http://localhost:38789/index.html?ui=P_0x7f9c741e8a10_3&reconnect=auto" class="pyvi…

## 3. Retrieve picked points and wrap into a `LabeledPointCloud`

Follows the pattern from `41_photogrammetric_optode_coregistration.ipynb` cells 26-28.

In [18]:
if INTERACTIVE:
    landmark_coordinates, landmark_labels = get_landmarks()
    landmark_coordinates = np.asarray(landmark_coordinates)
else:
    # Cached fallback: paste coordinates from a previous interactive run here.
    landmark_labels = ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa']
    landmark_coordinates = np.asarray([
        [0.0, 0.0, 0.0],   # Nz  -- replace
        [0.0, 0.0, 0.0],   # Iz  -- replace
        [0.0, 0.0, 0.0],   # Cz  -- replace
        [0.0, 0.0, 0.0],   # Lpa -- replace
        [0.0, 0.0, 0.0],   # Rpa -- replace
    ])

assert len(set(landmark_labels)) == 5, 'Need exactly 5 distinct landmarks'
assert set(landmark_labels) == {'Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'}, (
    f'Unexpected labels: {landmark_labels}'
)

landmarks = xr.DataArray(
    np.vstack(landmark_coordinates),
    dims=['label', 'digitized'],
    coords={
        'label': ('label', list(landmark_labels)),
        'type': ('label', [cdc.PointType.LANDMARK] * 5),
        'group': ('label', ['L'] * 5),
    },
).pint.quantify('mm')

display(landmarks)

Magnitude,[[65.80514234534597 -11.605171044162262 481.1656612460295] [182.6220307773288 116.67835657353872 431.57097619552144] [51.88537827784663 219.86486083549175 456.3823857394348] [64.75510597977373 78.3338583311504 566.3625948499965] [2.740867794190251 77.07375633870706 392.2325599877916]]
Units,millimeter


## 4. Normalize axes and isolate head

`normalize_axes` rotates the mesh around its X-axis so Y points anterior (toward
the face) and Z points left. We apply the same rotation to the 5 landmarks so
they stay consistent with the mesh. After this, the canonical frame is:
X=up, Y=anterior, Z=left.

In [ ]:
# Step 1: Mediapipe-compatible X-rotation so Y points anterior.
lm_raw = landmarks.pint.dequantify().values
idx = {lbl: i for i, lbl in enumerate(landmarks['label'].values)}
Nz_raw = lm_raw[idx['Nz']]

surface_n, Nz_n, R = normalize_axes(surface, Nz_raw)
landmarks_n = landmarks.pint.dequantify().copy(data=lm_raw @ R.T).pint.quantify()

# Head isolation (strips shoulders/background).
surface_n, _ = isolate_head(surface_n, Nz_n)
print(f'After isolate_head: {surface_n.nvertices:,} vertices')

# Step 2: Full 5-landmark alignment to canonical frame (X=up, Y=ant, Z=left).
surface_h, landmarks_n, R2 = align_axes_from_landmarks(surface_n, landmarks_n)
lm_n = landmarks_n.pint.dequantify().values
Nz, Iz, Cz, Lpa, Rpa = (lm_n[idx[lbl]] for lbl in ['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'])
ear_mid = 0.5 * (Lpa + Rpa)

print('\nAligned landmarks:')
verts = np.asarray(surface_h.mesh.vertices)
for lbl, pos in zip(['Nz', 'Iz', 'Cz', 'Lpa', 'Rpa'], [Nz, Iz, Cz, Lpa, Rpa]):
    d = np.linalg.norm(verts - pos, axis=1).min()
    print(f'  {lbl}: {pos}  (on mesh: {d:.2f} mm)')

## 5. Build the face-region mask from the 5 landmarks

Two parts:

1. **Cap boundary detection:** Scan upward from Nz along the midline. In each
   height bin, find the most anterior vertex (max Y). The forehead shows a smooth
   Y decrease; the cap edge shows a sudden Y drop. The X value where that drop
   first exceeds `CAP_GRADIENT_THRESHOLD` is `cap_x`.

2. **Mask rules** (in the aligned frame X=up, Y=anterior, Z=left):
   - `X < cap_x` (below the cap boundary -- covers face + forehead)
   - `Y > ear_mid_y` (anterior to the coronal ear plane)
   - `|Z - mid_z| < half_width` (within lateral width, ears excluded)

In [ ]:
# 1. Detect cap boundary from midline Y-profile.
verts = np.asarray(surface_h.mesh.vertices)
mid_z = 0.5 * (Lpa[2] + Rpa[2])

cap_x, profile_x, profile_y, profile_y_s = detect_cap_boundary(
    verts, Nz, Cz, ear_mid, mid_z,
    band_width=CAP_BAND_WIDTH_MM,
    bin_size=CAP_BIN_SIZE_MM,
    foot_grad_threshold=CAP_FOOT_GRAD_THRESHOLD,
)
print(f'Cap boundary detected at X = {cap_x:.1f} mm  '
      f'(Nz_x={Nz[0]:.1f}, Cz_x={Cz[0]:.1f})')

# Debug plot: Y-profile along midline (raw + smoothed) + gradient.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(profile_x, profile_y, 'b.-', alpha=0.4, label='raw max-Y')
ax1.plot(profile_x, profile_y_s, 'b-', lw=2, label='smoothed')
ax1.axvline(Nz[0], color='green', ls='--', label='Nz')
ax1.axvline(cap_x, color='red', ls='--', label=f'cap edge ({cap_x:.0f})')
ax1.axvline(Cz[0], color='cyan', ls='--', label='Cz')
ax1.set_xlabel('X (height, mm)')
ax1.set_ylabel('max Y (anterior, mm)')
ax1.set_title('Midline Y-profile')
ax1.legend()

grad = np.gradient(profile_y_s, profile_x)
ax2.plot(profile_x, grad, 'r.-')
ax2.axhline(CAP_FOOT_GRAD_THRESHOLD, color='purple', ls=':', label='foot threshold')
ax2.axvline(cap_x, color='red', ls='--', label='cap edge')
ax2.set_xlabel('X (height, mm)')
ax2.set_ylabel('dY/dX')
ax2.set_title('Gradient')
ax2.legend()
plt.tight_layout()
plt.show()

# 2. Build face+ear mask from landmarks.
mask, mask_info = face_mask_from_landmarks(
    verts, Nz, Iz, Cz, Lpa, Rpa,
    cap_x=cap_x,
    ear_delete_radius=EAR_DELETE_RADIUS_MM,
)

# 3. Preserve small spheres around each landmark point.
for lm in (Nz, Iz, Cz, Lpa, Rpa):
    near = np.linalg.norm(verts - lm, axis=1) < LANDMARK_KEEP_RADIUS
    mask[near] = False

# 4. Preserve a midline Nz strip upward to the cap boundary (for coregistration).
nasion_strip = (
    (verts[:, 0] >= Nz[0]) &
    (verts[:, 0] < cap_x) &
    (np.abs(verts[:, 2] - Nz[2]) < LANDMARK_KEEP_RADIUS) &
    (verts[:, 1] > ear_mid[1])
)
mask[nasion_strip] = False

print(f'Nasion strip preserved: {int(nasion_strip.sum()):,} vertices')
print(f'Mask: {mask.sum():,} / {len(mask):,} vertices '
      f'({100*mask.sum()/len(mask):.1f}%)')
print(f'Per-rule counts: {mask_info["counts"]}')

## 6. Visualize the anonymization mask

Red = will be deleted, white = kept. The 5 landmark spheres are overlaid for
orientation. This is the primary deliverable — if the red region hugs the face
without leaking onto ears / top-of-head / neck, the geometric rules are working.

In [ ]:
lm_colors = {'Nz': 'lime', 'Iz': 'magenta', 'Cz': 'cyan', 'Lpa': 'orange', 'Rpa': 'blue'}

pvplt = pv.Plotter()

vtk_surface = VTKSurface.from_trimeshsurface(surface_h)
pv_mesh = pv.wrap(vtk_surface.mesh)
pv_mesh['mask'] = mask.astype(float)
pvplt.add_mesh(
    pv_mesh, scalars='mask', cmap=['white', 'red'], clim=[0, 1],
    show_scalar_bar=False, opacity=1.0, smooth_shading=True,
)

for lbl, pos in zip(landmarks_n['label'].values, lm_n):
    c = lm_colors.get(lbl, 'yellow')
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos), color=c)
    pvplt.add_point_labels(
        [pos], [lbl], font_size=16, text_color=c, shape=None, always_visible=True,
    )

pvplt.add_text(
    f'S{SUBJECT_NUMBER} | mask: {mask.sum():,} / {len(mask):,} verts '
    f'({100*mask.sum()/len(mask):.1f}%)',
    position='upper_left', font_size=12,
)
pvplt.show()

## 7. Anonymize by deleting masked vertices

Drops every triangle that has any masked vertex, then strips unreferenced
vertices. Pure trimesh, no smoothing.

In [ ]:
surface_anon = delete_masked_vertices(surface_h, mask)
n_removed = surface_h.nvertices - surface_anon.nvertices
print(f'Original:    {surface_h.nvertices:,} verts, {surface_h.nfaces:,} faces')
print(f'Anonymized:  {surface_anon.nvertices:,} verts, {surface_anon.nfaces:,} faces')
print(f'Removed:     {n_removed:,} verts')

## 8. Before / after

In [ ]:
anon_vtk = trimesh_to_vtk_polydata(surface_anon.mesh)

pvplt = pv.Plotter(shape=(1, 2))

pvplt.subplot(0, 0)
cedalion.plots.plot_surface(pvplt, surface_h, opacity=1.0)
for lbl, pos in zip(landmarks_n['label'].values, lm_n):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos),
                   color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(f'S{SUBJECT_NUMBER} Original', position='upper_left', font_size=14)

pvplt.subplot(0, 1)
pvplt.add_mesh(pv.wrap(anon_vtk), rgb=True, smooth_shading=True)
for lbl, pos in zip(landmarks_n['label'].values, lm_n):
    pvplt.add_mesh(pv.Sphere(radius=4, center=pos),
                   color=lm_colors.get(lbl, 'yellow'))
pvplt.add_text(
    f'S{SUBJECT_NUMBER} Anonymized (-{n_removed:,} verts)',
    position='upper_left', font_size=14,
)

pvplt.link_views()
pvplt.show()

## 9. Validate

Sanity-check the result: vertex count dropped, no degenerate faces, landmarks
still reachable on the anonymized surface.

In [101]:
result = validate_anonymization(
    original_surface=surface_h,
    anonymized_surface=surface_anon,
    facial_mask=mask,
    protected_points=landmarks_n,
    tolerance=5.0 * cedalion.units.mm,
)
print(result)

ValidationResult(face_removed=True, mesh_valid=True, expected_vertices_removed=117152, actual_vertices_removed=117230, protected_points_intact=True, protected_point_max_deviation=0.3121421099641441, face_coverage_pct=23.709748253927778, passed=True, summary='PASSED — 117,230 vertices removed (23.7%), mesh valid, protected points within 5.0mm')


## Notes

- **Landmarks used:** 5 given (Nz, Iz, Cz, Lpa, Rpa), acquired via the existing
  upstream cedalion picker.
- **Mask thresholds:** `MARGIN_TOP_MM=0`, `MARGIN_BOTTOM_MM=20`, `LATERAL_SCALE=0.9`
  (tunable at the top of the notebook).
- **Anonymization method:** vertex deletion (drop faces touching masked vertices,
  strip unreferenced vertices). No Taubin smoothing. No MediaPipe.
- **Contract with the exposé:** this notebook assumes NAS/LPA/RPA are given and
  uses no DL face detection. Cz and Iz are also given and used as vertical bounds;
  they aren't strictly required but make the top/bottom cuts landmark-driven
  instead of mesh-extent-driven.
- **Relationship to notebook 47:** this is the cedalion-only alternative path.
  Notebook 47 detects Nz via MediaPipe and uses a face-oval contour for the
  upper mask bound; this notebook derives everything from the 5 given landmarks.